# Salidas estructuradas: que el modelo rellene tu formulario

**Facsímil 4 · La caja de herramientas** — capítulo 2 (APIs de modelos: mensajes, *streaming* y salidas
estructuradas).

Si vas a meter lo que dice un LLM en una base de datos, en una factura o en una decisión, no puedes
permitirte un «más o menos». En este cuaderno dejas de tratar la respuesta del modelo como un texto del
que «sacar» datos a mano y empiezas a **exigirle un formato fijo** que validas automáticamente. Vas a
definir un esquema, comprobar una respuesta buena, **cazar** una mala antes de que cause daño, y montar
el bucle de reintento que usan los sistemas serios. Las salidas estructuradas convierten un texto
simpático en un **dato fiable** o en un **error que se ve venir**.

### Qué vas a aprender
- Por qué «parsear» texto libre de un LLM es frágil, y qué lo resuelve.
- A definir un **esquema** (con Pydantic) que describe exactamente qué esperas y de qué tipo.
- A **validar** una respuesta y convertirla en un objeto con tipos (una fecha que es una fecha de
  verdad).
- A **rechazar** respuestas mal formadas con un mensaje claro, y a montar el **reintento** con el error.
- Por qué este patrón es la base de las *tools* de un agente (facsímil 5).

### Cuánto cuesta
Unos 12 minutos. CPU, sin claves (simulamos las respuestas del modelo).


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [1]:
# En Colab:  !pip install pydantic
from pydantic import BaseModel, field_validator, ValidationError
from datetime import date

class Factura(BaseModel):
    proveedor: str
    importe_eur: float
    fecha: date
    concepto: str

    @field_validator("importe_eur")
    @classmethod
    def positivo(cls, v):
        if v <= 0:
            raise ValueError("el importe debe ser positivo")
        return v

print("Esquema definido. Una factura VALIDA debe tener: proveedor, importe>0, fecha y concepto.")


Esquema definido. Una factura VALIDA debe tener: proveedor, importe>0, fecha y concepto.


## 1. El problema de «parsear» texto libre

Imagina que le pides a un LLM los datos de una factura y responde: «La factura es de Acme, por unos 50
euros, creo que de finales de mayo». ¿Cómo extraes de ahí el importe exacto y la fecha para tu base de
datos? Con expresiones regulares frágiles que se rompen en cuanto el modelo cambia una coma. Es el
mismo error que cometía la informática antes de las APIs: tratar datos como texto. La solución: **pedir
estructura** (JSON con un esquema) y **validarla**. Así el modelo rellena un formulario, y tú compruebas
que está bien antes de usarlo.


## 2. El caso: extraer una factura de un correo

Esta sería una respuesta **buena** del modelo, en JSON. La validamos contra el esquema: si pasa, ya es
un objeto con **tipos correctos** —la fecha es un objeto fecha, no un texto—, listo para guardar u
operar (calcular el mes, comparar, sumar importes).


In [2]:
respuesta_buena = '''{
  "proveedor": "Hosting Tarradellas S.L.",
  "importe_eur": 49.90,
  "fecha": "2026-05-31",
  "concepto": "Servidor dedicado mayo"
}'''

factura = Factura.model_validate_json(respuesta_buena)
print("Validada. Ya es un DATO, no un texto:")
print("  proveedor:", factura.proveedor)
print("  importe:", factura.importe_eur, "EUR (tipo", type(factura.importe_eur).__name__, ")")
print("  fecha:", factura.fecha, "-> tipo", type(factura.fecha).__name__)
print("  mes de la fecha (operable):", factura.fecha.month)


Validada. Ya es un DATO, no un texto:
  proveedor: Hosting Tarradellas S.L.
  importe: 49.9 EUR (tipo float )
  fecha: 2026-05-31 -> tipo date
  mes de la fecha (operable): 5


## 3. Cuando el modelo se equivoca (y lo hará)

Los LLM, a veces, devuelven un importe como texto, se inventan un campo o se dejan otro. Sin validación,
eso entra en tu base de datos y revienta tres pasos después, lejos de donde se originó el problema. Con
esquema, **salta aquí mismo**, con un mensaje claro de qué campo falló y por qué.


In [3]:
respuestas_malas = [
    ('importe como texto',   '{"proveedor": "Acme", "importe_eur": "cuarenta", "fecha": "2026-05-31", "concepto": "x"}'),
    ('importe negativo',     '{"proveedor": "Acme", "importe_eur": -10, "fecha": "2026-05-31", "concepto": "x"}'),
    ('falta la fecha',       '{"proveedor": "Acme", "importe_eur": 10, "concepto": "x"}'),
    ('fecha mal formada',    '{"proveedor": "Acme", "importe_eur": 10, "fecha": "31 de mayo", "concepto": "x"}'),
]
for motivo, r in respuestas_malas:
    try:
        Factura.model_validate_json(r); print(f"  [{motivo}] OK (no deberia)")
    except ValidationError as e:
        err = e.errors()[0]
        print(f"  RECHAZADA ({motivo}) -> campo '{err['loc'][0]}': {err['msg']}")


  RECHAZADA (importe como texto) -> campo 'importe_eur': Input should be a valid number, unable to parse string as a number
  RECHAZADA (importe negativo) -> campo 'importe_eur': Value error, el importe debe ser positivo
  RECHAZADA (falta la fecha) -> campo 'fecha': Field required
  RECHAZADA (fecha mal formada) -> campo 'fecha': Input should be a valid date or datetime, invalid character in year


**Eso es el valor del esquema.** Cada respuesta defectuosa se ha quedado en la puerta con un motivo
legible, *antes* de tocar nada: el importe que no es número, el negativo, el campo que falta, la fecha
mal escrita. Ninguna ha llegado a tu base de datos. El modelo propone; tu código valida y dispone.


## 4. El bucle de reintento

En un sistema real, si la validación falla, no te rindes: le **devuelves al modelo el error** y le pides
que lo corrija. Es el patrón estándar, porque los modelos fallan a veces pero suelen acertar al segundo
intento si les dices qué estuvo mal. Simulamos ese ida y vuelta.


In [4]:
def pedir_al_modelo(intento, ultimo_error=None):
    # Simulacion: el primer intento sale mal (coma decimal), el segundo corrige.
    if intento == 1:
        return '{"proveedor": "Acme", "importe_eur": "49,90", "fecha": "2026-05-31", "concepto": "x"}'
    return '{"proveedor": "Acme", "importe_eur": 49.90, "fecha": "2026-05-31", "concepto": "x"}'

ultimo_error = None
for intento in (1, 2, 3):
    try:
        f = Factura.model_validate_json(pedir_al_modelo(intento, ultimo_error))
        print(f"Intento {intento}: VALIDO -> {f.importe_eur} EUR. Guardado en base de datos.")
        break
    except ValidationError as e:
        ultimo_error = e.errors()[0]['msg']
        print(f"Intento {intento}: invalido ({ultimo_error}). Reintento devolviendo el error al modelo...")


Intento 1: invalido (Input should be a valid number, unable to parse string as a number). Reintento devolviendo el error al modelo...
Intento 2: VALIDO -> 49.9 EUR. Guardado en base de datos.


## 5. Más allá de la validación: tipos y reglas de negocio

Un esquema no solo comprueba tipos: puede codificar **reglas de negocio**. Añadimos una regla —la fecha
no puede ser futura— y un campo opcional con valor por defecto. Así el esquema se vuelve un contrato
completo de lo que es una factura aceptable en *tu* sistema, no solo un JSON bien formado.


In [5]:
from typing import Optional
class FacturaEstricta(Factura):
    iva_incluido: Optional[bool] = True       # opcional, por defecto True
    @field_validator("fecha")
    @classmethod
    def no_futura(cls, v):
        if v > date(2026, 6, 26):
            raise ValueError("la fecha no puede ser futura")
        return v

casos = [
    ('factura normal',  '{"proveedor":"Acme","importe_eur":10,"fecha":"2026-01-10","concepto":"x"}'),
    ('fecha futura',    '{"proveedor":"Acme","importe_eur":10,"fecha":"2027-01-10","concepto":"x"}'),
]
for motivo, r in casos:
    try:
        f = FacturaEstricta.model_validate_json(r)
        print(f"  [{motivo}] OK -> iva_incluido={f.iva_incluido} (valor por defecto aplicado)")
    except ValidationError as e:
        print(f"  [{motivo}] RECHAZADA -> {e.errors()[0]['msg']}")


  [factura normal] OK -> iva_incluido=True (valor por defecto aplicado)
  [fecha futura] RECHAZADA -> Value error, la fecha no puede ser futura


## 6. Pruébalo tú

1. **Añade un campo** `moneda` que solo acepte «EUR», «USD» o «GBP» (un validador). ¿Qué pasa si el
   modelo manda «euros»?
2. **Importe máximo:** rechaza facturas de más de 100.000 €. Los esquemas también ponen topes de
   negocio, no solo de tipo.
3. **Lista de líneas:** modela una factura con varias líneas (`items: list[Linea]`). Los esquemas
   anidan, igual que el JSON.
4. **Quita la validación** y mete una respuesta mala directamente en un `dict`. ¿Ves cómo el error
   aparecería mucho más tarde y más difícil de rastrear?


## 7. Errores comunes

- **Confiar en que el modelo «siempre» devuelve JSON válido.** No siempre. Valida, y ten un plan B
  (reintento, valor por defecto, error controlado).
- **Validar solo tipos y olvidar las reglas de negocio.** Un importe negativo o una fecha futura son
  JSON válidos pero datos inválidos. El esquema debe capturar ambas cosas.
- **No devolver el error al modelo en el reintento.** Si solo reintentas sin decirle qué falló, repetirá
  el mismo fallo. El error es la pista para que corrija.
- **Parsear con expresiones regulares.** Frágil y propenso a romperse. Pide estructura y valida.


## 8. Qué te llevas

- Pedir **salida estructurada** (un esquema) convierte la respuesta del modelo en un **dato con tipos**,
  no en un texto que hay que interpretar a mano.
- La validación **caza los fallos en el sitio**, con un mensaje claro, en vez de propagarlos a tu base de
  datos o a una decisión. Y puede codificar **reglas de negocio**, no solo tipos.
- El **reintento con el error** es el patrón estándar: los modelos fallan, el sistema se recupera.

**Para seguir:** este patrón es la base de las *tools* de un agente (facsímil 5), donde cada herramienta
tiene un contrato de entrada y salida que se valida igual antes de ejecutar nada.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*